In [1]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 1 (REBUILT): DATA PREPARATION — PROPHET MODEL
# Approach: Compare current values vs fixed 5Y baseline average
#
# Baseline: Jan 2021 → Dec 2025 (60 months)
# Current:  Jan 2026 → Jun 2026 (actual data)
# Forecast: Jul 2026 → Sep 2026 (Prophet will produce)
#
# For each indicator:
#   1. Load raw source data
#   2. Compute fixed 5Y baseline average
#   3. Compute deviation or ratio vs baseline
#   4. This deviation/ratio becomes the Prophet input
#
# Indicators:
#   STU:    deviation = current - baseline_avg
#   BDI:    ratio     = current / baseline_avg
#   Demand: ratio     = current / baseline_avg
#   PPI:    ratio     = current / baseline_avg
#   KSA:    deviation = current - baseline_avg (%)
#   Policy: rule-based — no baseline needed
#
# Output tables:
#   srm.prophet_ts_stu
#   srm.prophet_ts_demand
#   srm.prophet_ts_ppi
#   srm.prophet_ts_ksa
#   srm.prophet_ts_policy
#   srm.prophet_ts_bdi
#   srm.prophet_ts_all
#   srm.prophet_baseline_summary  ← new: baseline stats
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

COMMODITIES   = ["Wheat","Corn","Rice","Soybean"]
BASELINE_START = pd.Timestamp("2021-01-01")
BASELINE_END   = pd.Timestamp("2025-12-01")
SCORING_END    = pd.Timestamp("2026-06-01")

STRING_COLS = [
    "ksa_top1_country","ksa_top1_country_lag1",
    "ksa_top3_countries","top_buyer_country",
    "individually_tracked_countries"
]

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    spark.createDataFrame(df_pandas) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name, drop_strings=False):
    df_spark = spark.table(f"srm.{table_name}")
    if drop_strings:
        drop_cols = [c for c in STRING_COLS if c in df_spark.columns]
        if drop_cols:
            df_spark = df_spark.drop(*drop_cols)
    df = df_spark.toPandas()
    if "year_month" in df.columns:
        df["year_month"] = pd.to_datetime(df["year_month"])
    return df

print("=== Notebook 1 (Rebuilt): Data Preparation for Prophet ===")
print(f"Baseline period: {BASELINE_START.date()} → {BASELINE_END.date()}")
print(f"Current period:  Jan 2026 → {SCORING_END.date()}")
print(f"Approach:        Fixed 5Y baseline average comparison")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 3, Finished, Available, Finished, False)

=== Notebook 1 (Rebuilt): Data Preparation for Prophet ===
Baseline period: 2021-01-01 → 2025-12-01
Current period:  Jan 2026 → 2026-06-01
Approach:        Fixed 5Y baseline average comparison


In [2]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD ALL RAW SOURCE TABLES
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading raw source tables ===")

df_wasde  = load_table("features_wasde")
df_demand = load_table("features_demand", drop_strings=True)
df_ksa    = load_table("features_ksa",    drop_strings=True)
df_bdi    = load_table("features_bdi")
df_policy = load_table("features_policy", drop_strings=True)

print(f"WASDE:   {df_wasde.shape}  | {df_wasde['year_month'].min().date()} → {df_wasde['year_month'].max().date()}")
print(f"Demand:  {df_demand.shape} | {df_demand['year_month'].min().date()} → {df_demand['year_month'].max().date()}")
print(f"KSA:     {df_ksa.shape}    | {df_ksa['year_month'].min().date()} → {df_ksa['year_month'].max().date()}")
print(f"BDI:     {df_bdi.shape}    | {df_bdi['year_month'].min().date()} → {df_bdi['year_month'].max().date()}")
print(f"Policy:  {df_policy.shape} | {df_policy['year_month'].min().date()} → {df_policy['year_month'].max().date()}")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 4, Finished, Available, Finished, False)


=== Step 1: Loading raw source tables ===
WASDE:   (264, 21)  | 2021-01-01 → 2026-06-01
Demand:  (223, 48) | 2021-01-01 → 2026-05-01
KSA:     (240, 15)    | 2021-01-01 → 2025-12-01
BDI:     (90, 18)    | 2019-01-01 → 2026-06-01
Policy:  (240, 21) | 2021-01-01 → 2025-12-01


In [3]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: STU RATIO TIME SERIES
# Raw:       stu_ratio (monthly %)
# Baseline:  mean of stu_ratio Jan2021-Dec2025 per commodity
# Signal:    deviation = current - baseline_avg
# Direction: negative = below average = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: STU ratio — deviation from 5Y baseline ===")

stu_rows     = []
stu_baseline = {}

for commodity in COMMODITIES:
    dc = df_wasde[df_wasde["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    # Fill any gaps in time series
    full_range = pd.date_range(dc["year_month"].min(),
                               dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"] = commodity
    dc["stu_ratio"]  = dc["stu_ratio"].ffill()
    dc = dc.reset_index()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "stu_ratio"].mean()
    stu_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg (Jan2021-Dec2025): {baseline_avg:.2f}%")
    print(f"  Latest value:                      {dc['stu_ratio'].iloc[-1]:.2f}%")
    print(f"  Deviation from baseline:           {dc['stu_ratio'].iloc[-1] - baseline_avg:.2f}%")

    # Compute deviation for each month
    dc["baseline_avg"] = baseline_avg
    dc["y"]            = dc["stu_ratio"] - baseline_avg  # deviation
    dc["y_raw"]        = dc["stu_ratio"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            stu_rows.append({
                "ds":           row["year_month"],
                "y":            round(row["y"], 4),       # deviation from baseline
                "y_raw":        round(row["y_raw"], 4),   # actual STU value
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "stu_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_stu_ts = pd.DataFrame(stu_rows)
df_stu_ts = df_stu_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nSTU deviation time series: {df_stu_ts.shape}")
print(f"\nDeviation stats (positive = above avg = good, negative = below avg = risk):")
print(df_stu_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_stu_ts, "prophet_ts_stu")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 5, Finished, Available, Finished, False)


=== Step 2: STU ratio — deviation from 5Y baseline ===

Wheat:
  5Y baseline avg (Jan2021-Dec2025): 26.86%
  Latest value:                      26.57%
  Deviation from baseline:           -0.29%

Corn:
  5Y baseline avg (Jan2021-Dec2025): 21.42%
  Latest value:                      18.38%
  Deviation from baseline:           -3.04%

Rice:
  5Y baseline avg (Jan2021-Dec2025): 30.84%
  Latest value:                      31.91%
  Deviation from baseline:           1.07%

Soybean:
  5Y baseline avg (Jan2021-Dec2025): 19.63%
  Latest value:                      19.82%
  Deviation from baseline:           0.19%

STU deviation time series: (264, 7)

Deviation stats (positive = above avg = good, negative = below avg = risk):
           count   mean    std    min    25%    50%    75%    max
commodity                                                        
Corn        66.0 -0.219  1.379 -3.185 -1.040  0.280  0.845  1.605
Rice        66.0  0.082  1.299 -2.045 -0.980  0.045  1.042  3.035
Soybean 

In [4]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: BDI TIME SERIES
# Raw:       bdi (monthly average index value)
# Baseline:  mean of bdi Jan2021-Dec2025
# Signal:    ratio = current / baseline_avg
# Direction: ratio > 1 = freight elevated = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 3: BDI — ratio vs 5Y baseline ===")

df_bdi_clean = df_bdi.sort_values("year_month").reset_index(drop=True)
df_bdi_clean["year_month"] = pd.to_datetime(df_bdi_clean["year_month"])

# Compute fixed 5Y baseline average
bdi_baseline_mask = (
    (df_bdi_clean["year_month"] >= BASELINE_START) &
    (df_bdi_clean["year_month"] <= BASELINE_END)
)
bdi_baseline_avg = df_bdi_clean.loc[bdi_baseline_mask, "bdi"].mean()

print(f"BDI 5Y baseline avg (Jan2021-Dec2025): {bdi_baseline_avg:.0f}")
print(f"BDI latest value (Jun 2026):           {df_bdi_clean['bdi'].iloc[-1]:.0f}")
print(f"BDI ratio vs baseline:                 {df_bdi_clean['bdi'].iloc[-1]/bdi_baseline_avg:.3f}")
print(f"Interpretation:                        {((df_bdi_clean['bdi'].iloc[-1]/bdi_baseline_avg-1)*100):.0f}% above 5Y average")

# Compute ratio for each month
df_bdi_clean["baseline_avg"] = bdi_baseline_avg
df_bdi_clean["y"]            = df_bdi_clean["bdi"] / bdi_baseline_avg  # ratio
df_bdi_clean["y_raw"]        = df_bdi_clean["bdi"]

bdi_rows = []
for commodity in COMMODITIES:
    for _, row in df_bdi_clean.iterrows():
        bdi_rows.append({
            "ds":           row["year_month"],
            "y":            round(row["y"], 4),       # ratio vs baseline
            "y_raw":        round(row["y_raw"], 2),   # actual BDI
            "baseline_avg": round(bdi_baseline_avg, 2),
            "commodity":    commodity,
            "indicator":    "bdi_ratio",
            "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
        })

df_bdi_ts = pd.DataFrame(bdi_rows).dropna(subset=["y"])
df_bdi_ts = df_bdi_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nBDI ratio time series: {df_bdi_ts.shape}")
print(f"\nBDI ratio stats (1.0 = at baseline, >1 = elevated):")
print(df_bdi_ts[df_bdi_ts["commodity"]=="Wheat"]["y"].describe().round(3))

save_to_lakehouse(df_bdi_ts, "prophet_ts_bdi")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 6, Finished, Available, Finished, False)


=== Step 3: BDI — ratio vs 5Y baseline ===
BDI 5Y baseline avg (Jan2021-Dec2025): 1938
BDI latest value (Jun 2026):           2864
BDI ratio vs baseline:                 1.478
Interpretation:                        48% above 5Y average

BDI ratio time series: (360, 7)

BDI ratio stats (1.0 = at baseline, >1 = elevated):
count    90.000
mean      0.913
std       0.407
min       0.238
25%       0.676
50%       0.866
75%       1.068
max       2.488
Name: y, dtype: float64
✓ srm.prophet_ts_bdi: 360 rows saved


In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: DEMAND PRESSURE TIME SERIES
# Raw:       driver_total_mt (total import volume MT
#            from all major tracked buyers per commodity)
# Baseline:  mean of driver_total_mt Jan2021-Dec2025
# Signal:    ratio = current / baseline_avg
# Direction: ratio > 1 = demand elevated = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Demand — ratio vs 5Y baseline ===")

demand_rows     = []
demand_baseline = {}

for commodity in COMMODITIES:
    dc = df_demand[df_demand["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    # Fill any month gaps with 0 (no data = no tracked buyer activity)
    full_range = pd.date_range(dc["year_month"].min(),
                               dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"]      = commodity
    dc["driver_total_mt"] = dc["driver_total_mt"].fillna(0)
    dc = dc.reset_index()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "driver_total_mt"].mean()
    demand_baseline[commodity] = round(baseline_avg, 2)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg imports: {baseline_avg/1e6:.2f}M MT/month")
    print(f"  Latest value:            {dc['driver_total_mt'].iloc[-1]/1e6:.2f}M MT")

    # Compute ratio — handle division by zero
    if baseline_avg > 0:
        dc["y"] = dc["driver_total_mt"] / baseline_avg
    else:
        dc["y"] = 1.0

    dc["y_raw"]        = dc["driver_total_mt"]
    dc["baseline_avg"] = baseline_avg

    print(f"  Latest ratio vs baseline: {dc['y'].iloc[-1]:.3f}")

    for _, row in dc.iterrows():
        demand_rows.append({
            "ds":           row["year_month"],
            "y":            round(float(row["y"]), 4),
            "y_raw":        round(float(row["y_raw"]), 2),
            "baseline_avg": round(baseline_avg, 2),
            "commodity":    commodity,
            "indicator":    "demand_ratio",
            "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
        })

df_demand_ts = pd.DataFrame(demand_rows)
df_demand_ts = df_demand_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nDemand ratio time series: {df_demand_ts.shape}")
print(f"\nDemand ratio stats (1.0 = at baseline, >1 = elevated):")
print(df_demand_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_demand_ts, "prophet_ts_demand")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 7, Finished, Available, Finished, False)


=== Step 4: Demand — ratio vs 5Y baseline ===

Wheat:
  5Y baseline avg imports: 5.97M MT/month
  Latest value:            0.59M MT
  Latest ratio vs baseline: 0.099

Corn:
  5Y baseline avg imports: 6.98M MT/month
  Latest value:            0.19M MT
  Latest ratio vs baseline: 0.027

Rice:
  5Y baseline avg imports: 1.26M MT/month
  Latest value:            0.11M MT
  Latest ratio vs baseline: 0.085

Soybean:
  5Y baseline avg imports: 27.59M MT/month
  Latest value:            21.47M MT
  Latest ratio vs baseline: 0.778

Demand ratio time series: (223, 7)

Demand ratio stats (1.0 = at baseline, >1 = elevated):
           count   mean    std    min    25%    50%    75%    max
commodity                                                        
Corn        52.0  0.961  0.278  0.027  0.817  0.958  1.138  1.710
Rice        53.0  0.959  0.270  0.085  0.862  0.956  1.088  1.497
Soybean     65.0  0.982  0.317  0.388  0.776  0.996  1.174  1.742
Wheat       53.0  0.956  0.227  0.100  0.822  0.9

In [6]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: PPI TIME SERIES
# Raw:       ppi_base from features_wasde
#            ppi_base = (production / 12m_rolling_avg) × 100
#            value of 100 = production at 12m average
# Baseline:  mean of ppi_base Jan2021-Dec2025 per commodity
# Signal:    ratio = current / baseline_avg
# Direction: ratio < 1 = production below baseline = risk
# Note:      ppi_base already normalised around 100
#            baseline_avg will be close to 100
#            ratio will be close to 1
#            Deviation = current - baseline is more meaningful
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: PPI — deviation vs 5Y baseline ===")

ppi_rows     = []
ppi_baseline = {}

for commodity in COMMODITIES:
    dc = df_wasde[df_wasde["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])
    dc["ppi_base"]   = dc["ppi_base"].ffill()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "ppi_base"].mean()
    ppi_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg PPI:   {baseline_avg:.2f}")
    print(f"  Latest PPI value:      {dc['ppi_base'].iloc[-1]:.2f}")
    print(f"  Deviation:             {dc['ppi_base'].iloc[-1] - baseline_avg:.2f}")
    print(f"  Interpretation:        production {'above' if dc['ppi_base'].iloc[-1] >= baseline_avg else 'below'} 5Y average")

    # Use deviation = current - baseline
    # Positive = production above baseline = good
    # Negative = production below baseline = risk
    dc["baseline_avg"] = baseline_avg
    dc["y"]            = dc["ppi_base"] - baseline_avg  # deviation
    dc["y_raw"]        = dc["ppi_base"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            ppi_rows.append({
                "ds":           row["year_month"],
                "y":            round(float(row["y"]), 4),
                "y_raw":        round(float(row["y_raw"]), 4),
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "ppi_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_ppi_ts = pd.DataFrame(ppi_rows)
df_ppi_ts = df_ppi_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nPPI deviation time series: {df_ppi_ts.shape}")
print(f"\nPPI deviation stats (positive = above avg = good):")
print(df_ppi_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_ppi_ts, "prophet_ts_ppi")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 8, Finished, Available, Finished, False)


=== Step 5: PPI — deviation vs 5Y baseline ===

Wheat:
  5Y baseline avg PPI:   100.62
  Latest PPI value:      99.35
  Deviation:             -1.27
  Interpretation:        production below 5Y average

Corn:
  5Y baseline avg PPI:   101.01
  Latest PPI value:      101.17
  Deviation:             0.16
  Interpretation:        production above 5Y average

Rice:
  5Y baseline avg PPI:   100.86
  Latest PPI value:      99.47
  Deviation:             -1.39
  Interpretation:        production below 5Y average

Soybean:
  5Y baseline avg PPI:   101.51
  Latest PPI value:      103.28
  Deviation:             1.77
  Interpretation:        production above 5Y average

PPI deviation time series: (216, 7)

PPI deviation stats (positive = above avg = good):
           count   mean    std    min    25%    50%    75%    max
commodity                                                        
Corn        54.0  0.148  2.339 -3.585 -1.721  0.398  2.094  3.869
Rice        54.0 -0.080  1.055 -2.423 -0.734 

In [7]:
# ══════════════════════════════════════════════════════════════════
# STEP 6: KSA CONCENTRATION TIME SERIES
# Raw:       ksa_top3_share (decimal e.g. 0.97 = 97%)
#            Annual data — same value repeated each month
# Baseline:  mean of ksa_top3_share Jan2021-Dec2025 per commodity
# Signal:    deviation = current% - baseline_avg%
# Direction: positive = more concentrated than avg = risk
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 6: KSA concentration — deviation vs 5Y baseline ===")

ksa_rows     = []
ksa_baseline = {}

for commodity in COMMODITIES:
    dc = df_ksa[df_ksa["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])

    # Convert to percentage
    dc["ksa_top3_pct"] = dc["ksa_top3_share"] * 100
    dc["ksa_top3_pct"] = dc["ksa_top3_pct"].ffill()

    # Fill any month gaps
    full_range = pd.date_range(dc["year_month"].min(),
                               dc["year_month"].max(), freq="MS")
    dc = dc.set_index("year_month").reindex(full_range)
    dc.index.name = "year_month"
    dc["commodity"]    = commodity
    dc["ksa_top3_pct"] = dc["ksa_top3_pct"].ffill()
    dc = dc.reset_index()

    # Compute fixed 5Y baseline average
    baseline_mask = (
        (dc["year_month"] >= BASELINE_START) &
        (dc["year_month"] <= BASELINE_END)
    )
    baseline_avg = dc.loc[baseline_mask, "ksa_top3_pct"].mean()
    ksa_baseline[commodity] = round(baseline_avg, 4)

    print(f"\n{commodity}:")
    print(f"  5Y baseline avg top3 share: {baseline_avg:.1f}%")
    print(f"  Latest value:               {dc['ksa_top3_pct'].iloc[-1]:.1f}%")
    print(f"  Deviation from baseline:    {dc['ksa_top3_pct'].iloc[-1] - baseline_avg:+.1f}%")
    print(f"  Interpretation:             concentration {'rising above' if dc['ksa_top3_pct'].iloc[-1] > baseline_avg else 'below'} 5Y average")

    # Deviation = current - baseline
    dc["baseline_avg"] = baseline_avg
    dc["y"]            = dc["ksa_top3_pct"] - baseline_avg  # deviation in %
    dc["y_raw"]        = dc["ksa_top3_pct"]

    for _, row in dc.iterrows():
        if pd.notna(row["y"]):
            ksa_rows.append({
                "ds":           row["year_month"],
                "y":            round(float(row["y"]), 4),
                "y_raw":        round(float(row["y_raw"]), 4),
                "baseline_avg": round(baseline_avg, 4),
                "commodity":    commodity,
                "indicator":    "ksa_deviation",
                "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
            })

df_ksa_ts = pd.DataFrame(ksa_rows)
df_ksa_ts = df_ksa_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"\nKSA deviation time series: {df_ksa_ts.shape}")
print(f"\nKSA deviation stats (positive = more concentrated than baseline = risk):")
print(df_ksa_ts.groupby("commodity")["y"].describe().round(3))

save_to_lakehouse(df_ksa_ts, "prophet_ts_ksa")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 9, Finished, Available, Finished, False)


=== Step 6: KSA concentration — deviation vs 5Y baseline ===

Wheat:
  5Y baseline avg top3 share: 81.9%
  Latest value:               87.1%
  Deviation from baseline:    +5.2%
  Interpretation:             concentration rising above 5Y average

Corn:
  5Y baseline avg top3 share: 96.8%
  Latest value:               97.4%
  Deviation from baseline:    +0.6%
  Interpretation:             concentration rising above 5Y average

Rice:
  5Y baseline avg top3 share: 92.4%
  Latest value:               93.1%
  Deviation from baseline:    +0.8%
  Interpretation:             concentration rising above 5Y average

Soybean:
  5Y baseline avg top3 share: 98.6%
  Latest value:               100.0%
  Deviation from baseline:    +1.4%
  Interpretation:             concentration rising above 5Y average

KSA deviation time series: (240, 7)

KSA deviation stats (positive = more concentrated than baseline = risk):
           count  mean     std     min    25%    50%    75%     max
commodity             

In [8]:
# ══════════════════════════════════════════════════════════════════
# STEP 7: POLICY TIME SERIES
# Rule-based — no 5Y avg applicable
# Carry forward as-is from features_policy
# policy_risk_score_weighted already 0-100
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 7: Policy — rule-based (no 5Y avg) ===")

policy_rows = []

for commodity in COMMODITIES:
    dc = df_policy[df_policy["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)
    dc["year_month"] = pd.to_datetime(dc["year_month"])
    dc["policy_risk_score_weighted"] = dc["policy_risk_score_weighted"].fillna(0)

    # No baseline comparison — policy is event-driven
    # Use raw score directly
    for _, row in dc.iterrows():
        policy_rows.append({
            "ds":           row["year_month"],
            "y":            round(float(row["policy_risk_score_weighted"]), 4),
            "y_raw":        round(float(row["policy_risk_score_weighted"]), 4),
            "baseline_avg": np.nan,   # not applicable
            "commodity":    commodity,
            "indicator":    "policy_risk_weighted",
            "data_period":  "baseline" if row["year_month"] <= BASELINE_END else "current"
        })

df_policy_ts = pd.DataFrame(policy_rows)
df_policy_ts = df_policy_ts.sort_values(["commodity","ds"]).reset_index(drop=True)

print(f"Policy time series: {df_policy_ts.shape}")
print(f"\nPolicy score stats:")
print(df_policy_ts.groupby("commodity")["y"].describe().round(2))

save_to_lakehouse(df_policy_ts, "prophet_ts_policy")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 10, Finished, Available, Finished, False)


=== Step 7: Policy — rule-based (no 5Y avg) ===
Policy time series: (240, 7)

Policy score stats:
           count   mean    std  min   25%    50%    75%    max
commodity                                                     
Corn        60.0   3.22   2.21  0.0  0.00   3.66   5.03   6.42
Rice        60.0  26.03  27.11  0.0  1.44  18.80  37.03  92.80
Soybean     60.0   2.86   2.05  0.0  0.00   3.01   3.92   6.26
Wheat       60.0  16.79  11.94  0.0  8.90  17.11  23.96  37.92
✓ srm.prophet_ts_policy: 240 rows saved


In [9]:
# ══════════════════════════════════════════════════════════════════
# STEP 8: BASELINE SUMMARY TABLE
# Document all baseline values for reference
# Critical for stakeholder transparency
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 8: Baseline summary ===")

baseline_rows = []

for commodity in COMMODITIES:
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "stu_ratio",
        "baseline_avg":       stu_baseline[commodity],
        "unit":               "% STU",
        "interpretation":     "5Y avg global stock-to-use ratio",
        "direction":          "deviation = current - baseline (negative = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "bdi_ratio",
        "baseline_avg":       round(bdi_baseline_avg, 2),
        "unit":               "BDI index",
        "interpretation":     "5Y avg BDI value (same for all commodities)",
        "direction":          "ratio = current / baseline (>1 = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "demand_ratio",
        "baseline_avg":       demand_baseline[commodity],
        "unit":               "MT/month",
        "interpretation":     "5Y avg total tracked buyer import volume",
        "direction":          "ratio = current / baseline (>1 = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "ppi_deviation",
        "baseline_avg":       ppi_baseline[commodity],
        "unit":               "PPI index",
        "interpretation":     "5Y avg production potential index",
        "direction":          "deviation = current - baseline (negative = risk)"
    })
    baseline_rows.append({
        "commodity":          commodity,
        "indicator":          "ksa_deviation",
        "baseline_avg":       ksa_baseline[commodity],
        "unit":               "% top3 share",
        "interpretation":     "5Y avg KSA top3 import concentration",
        "direction":          "deviation = current - baseline (positive = risk)"
    })

df_baseline = pd.DataFrame(baseline_rows)

print("\nBaseline values (Jan 2021 — Dec 2025):")
print(f"\n{'Commodity':<10} {'Indicator':<20} {'Baseline Avg':>14} {'Unit':<15}")
print("-" * 65)
for _, row in df_baseline.iterrows():
    print(f"{row['commodity']:<10} {row['indicator']:<20} {row['baseline_avg']:>14.3f} {row['unit']:<15}")

save_to_lakehouse(df_baseline, "prophet_baseline_summary")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 11, Finished, Available, Finished, False)


=== Step 8: Baseline summary ===

Baseline values (Jan 2021 — Dec 2025):

Commodity  Indicator              Baseline Avg Unit           
-----------------------------------------------------------------
Wheat      stu_ratio                    26.862 % STU          
Wheat      bdi_ratio                  1937.670 BDI index      
Wheat      demand_ratio            5968140.430 MT/month       
Wheat      ppi_deviation               100.620 PPI index      
Wheat      ksa_deviation                81.856 % top3 share   
Corn       stu_ratio                    21.425 % STU          
Corn       bdi_ratio                  1937.670 BDI index      
Corn       demand_ratio            6982974.530 MT/month       
Corn       ppi_deviation               101.010 PPI index      
Corn       ksa_deviation                96.762 % top3 share   
Rice       stu_ratio                    30.845 % STU          
Rice       bdi_ratio                  1937.670 BDI index      
Rice       demand_ratio            12557

In [10]:
# ══════════════════════════════════════════════════════════════════
# STEP 9: COMBINED TABLE
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 9: Building combined time series table ===")

df_all_ts = pd.concat([
    df_stu_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_bdi_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_demand_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_ppi_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_ksa_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
    df_policy_ts[["ds","y","y_raw","baseline_avg","commodity","indicator","data_period"]],
], ignore_index=True)

df_all_ts = df_all_ts.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

save_to_lakehouse(df_all_ts, "prophet_ts_all")

print(f"Combined table: {df_all_ts.shape}")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 12, Finished, Available, Finished, False)


=== Step 9: Building combined time series table ===
✓ srm.prophet_ts_all: 1543 rows saved
Combined table: (1543, 7)


In [11]:
# ══════════════════════════════════════════════════════════════════
# STEP 10: VALIDATION SUMMARY
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 10: Validation summary ===")

print(f"\n{'Indicator':<22} {'Commodity':<10} {'Baseline':>10} {'Current':>10} {'Signal':>12} {'Risk Direction'}")
print("-" * 80)

current_month = pd.Timestamp("2026-06-01")

indicator_map = {
    "stu_deviation":      df_stu_ts,
    "bdi_ratio":          df_bdi_ts,
    "demand_ratio":       df_demand_ts,
    "ppi_deviation":      df_ppi_ts,
    "ksa_deviation":      df_ksa_ts,
    "policy_risk_weighted": df_policy_ts
}

for indicator, df_ts in indicator_map.items():
    for commodity in COMMODITIES:
        dc = df_ts[df_ts["commodity"]==commodity]
        current_row = dc[dc["ds"] <= current_month].sort_values("ds").tail(1)
        if current_row.empty:
            continue

        y_current  = current_row["y"].values[0]
        y_raw      = current_row["y_raw"].values[0]
        baseline   = current_row["baseline_avg"].values[0] if "baseline_avg" in current_row.columns else np.nan

        if indicator in ["stu_deviation","ppi_deviation"]:
            risk_dir = "↓ Risk" if y_current < 0 else "✓ OK"
            signal   = f"{y_current:+.2f}"
        elif indicator in ["bdi_ratio","demand_ratio"]:
            risk_dir = "↑ Risk" if y_current > 1.0 else "✓ OK"
            signal   = f"{y_current:.3f}x"
        elif indicator == "ksa_deviation":
            risk_dir = "↑ Risk" if y_current > 0 else "✓ OK"
            signal   = f"{y_current:+.2f}%"
        else:
            risk_dir = "↑ Risk" if y_current > 30 else "✓ OK"
            signal   = f"{y_current:.1f}"

        print(f"{indicator:<22} {commodity:<10} {baseline:>10.2f} {y_raw:>10.2f} {signal:>12} {risk_dir}")

print(f"""
Signal interpretation:
  STU deviation:  negative = stocks below 5Y avg = risk
  BDI ratio:      > 1.0    = freight above 5Y avg = risk
  Demand ratio:   > 1.0    = imports above 5Y avg = risk
  PPI deviation:  negative = production below 5Y avg = risk
  KSA deviation:  positive = more concentrated than 5Y avg = risk
  Policy:         > 30     = active restrictions = risk

Baseline period: Jan 2021 → Dec 2025 (fixed)
""")

print("=== NOTEBOOK 1 COMPLETE ===")
print("Next: Notebook 2 — Prophet Models (retrain with new signal definitions)")

StatementMeta(, e91f1974-f069-49c1-8033-9613ff204d20, 13, Finished, Available, Finished, False)


=== Step 10: Validation summary ===

Indicator              Commodity    Baseline    Current       Signal Risk Direction
--------------------------------------------------------------------------------
stu_deviation          Wheat           26.86      26.57        -0.29 ↓ Risk
stu_deviation          Corn            21.42      18.38        -3.04 ↓ Risk
stu_deviation          Rice            30.84      31.91        +1.06 ✓ OK
stu_deviation          Soybean         19.63      19.82        +0.19 ✓ OK
bdi_ratio              Wheat         1937.67    2863.73       1.478x ↑ Risk
bdi_ratio              Corn          1937.67    2863.73       1.478x ↑ Risk
bdi_ratio              Rice          1937.67    2863.73       1.478x ↑ Risk
bdi_ratio              Soybean       1937.67    2863.73       1.478x ↑ Risk
demand_ratio           Wheat      5968140.43  593644.52       0.100x ✓ OK
demand_ratio           Corn       6982974.53  186582.55       0.027x ✓ OK
demand_ratio           Rice       1255743.72 